# Table A — Step 2+3+4: DINOv3-raw / JOCCH Geometry Filter + VGGT + Pose Scoring

Runs the DINOv3-raw and JOCCH retrieval manifests (from `retrieval_ablation_colab.ipynb`)
through the rest of the pipeline — `geometry_filter.py` (ASpanFormer) → `vggt_signals.py`
→ `pose_scoring.py` — the same way SSCD and CLIP were, in
`geometry_filter_ablation_colab.ipynb` + `vggt_ablation_colab.ipynb`, but with an enhanced
skip optimization.

**Two independent, additive skip/reuse sources**, both consulted by `geometry_filter_cached.py`:
1. `--cache-manifests` (existing): the reconstructed original DINO-production ASpanFormer
   record. Failure-only — pairs already confirmed `aspan_pass=False` there skip ASpanFormer.
   No real sidecars exist for that source's passes, so passes are always freshly run.
2. `--reuse-manifests` (new, this notebook): a consolidated manifest built by
   `build_aspan_reuse_manifest.py` from SSCD's and CLIP's own **completed, sidecar-bearing**
   geometry-filter runs. Unlike (1), this can skip **and reuse** confirmed passes too — the
   real sidecar is referenced in place (no copy needed), since ASpanFormer's result for a
   given pair of images doesn't depend on which retrieval model surfaced it. **Rebuilt before
   each of the four model×shard stages below**, with the source list growing each time, so
   later stages also benefit from sidecars this same notebook run already produced (e.g.
   JOCCH reusing DINOv3-raw's own freshly-computed sidecars, not just the pre-existing
   SSCD/CLIP data).

Every stage syncs its output to Drive immediately after it completes, not once at the end.

---

**Before running**, upload to Drive at `[DRIVE_ROOT]/scripts/`:
```
scripts/
    geometry_filter.py
    geometry_filter_cached.py          (extended with --reuse-manifests)
    build_aspan_reuse_manifest.py
    vggt_signals.py
    pose_scoring.py
    ml-aspanformer/
    _local/
        aspan_all_manifest_shard1_reconstructed.jsonl
        aspan_all_manifest_shard2_reconstructed.jsonl
```

**Inputs**:
```
ablation_outputs/retrieval/
    dinov3raw_retrieval_manifest.jsonl
    jocch_retrieval_manifest.jsonl      ← may not exist yet; those cells will say so and
                                           skip gracefully rather than fail
ablation_outputs/geometry_filter/
    sscd_shard1/   (aspan_all_manifest.jsonl + aspan_sidecars/, from geometry_filter_ablation_colab.ipynb)
    sscd_shard2/   (may not exist yet)
    clip_shard1/
    clip_shard2/   (may not exist yet)
```

**Outputs** (written to Drive incrementally, after every stage):
```
ablation_outputs/geometry_filter/
    aspan_reuse_manifest.jsonl                 ← rebuilt and re-synced before each stage
    dinov3raw_shard1/  clip_shard2/-style structure (aspan_all_manifest.jsonl, vggt_candidates_manifest.jsonl, aspan_sidecars/)
    dinov3raw_shard2/
    jocch_shard1/
    jocch_shard2/
ablation_outputs/vggt/
    dinov3raw_shard1/  (vggt_judged_manifest.jsonl, true_matches_manifest.jsonl, vggt_judge_summary.json, pose_scored_manifest.jsonl, pose_scoring_summary.json)
    dinov3raw_shard2/
    jocch_shard1/
    jocch_shard2/
```

In [ ]:
# Step 0 — Install dependencies and verify GPU
# einops pin: ml-aspanformer needs einops<0.6 (>=0.6 removed _torch_specific it depends on).
# opencv upgrade: Colab's opencv wheel is built against numpy 1.x; PyTorch 2.11+ pins numpy
# 2.x. opencv-python-headless>=4.9.0 added numpy 2.x ABI support (needed for vggt_signals.py).
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "einops==0.3.0", "-q"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "opencv-python-headless>=4.9.0", "-q"], check=True)

import torch, cv2, numpy as np
print(f"PyTorch : {torch.__version__}")
print(f"numpy   : {np.__version__}")
print(f"cv2     : {cv2.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:   {torch.cuda.get_device_name(0)}")
    print(f"VRAM:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU — switch runtime to GPU before continuing.")

In [ ]:
# Step 1 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 2 — Configure paths
# ── EDIT THESE ──────────────────────────────────────────────────────────────
from pathlib import Path
import json, shutil, sys, os

DRIVE_ROOT   = Path('/content/drive/MyDrive/ImageSimilarity')
SCRIPTS_DIR  = DRIVE_ROOT / 'scripts'
ASPANPATH    = '/content/ml-aspanformer'
WEIGHTS_PATH = DRIVE_ROOT / 'weights' / 'outdoor.ckpt'
CONFIG_PATH  = '/content/ml-aspanformer/configs/aspan/outdoor/aspan_test.py'
VGGT_CHECKPOINT = DRIVE_ROOT / 'weights' / 'vggt_omega_1b_512.pt'  # adjust filename if needed

SOURCE_ZIP = DRIVE_ROOT / 'workingsourcecrops.zip'
TARGET_ZIP = DRIVE_ROOT / 'working_targetcrops.zip'
LOCAL_SOURCE_DIR = Path('./source')
LOCAL_TARGET_DIR = Path('./target')

# Retrieval manifests (from retrieval_ablation_colab.ipynb)
RETRIEVAL_OUT = DRIVE_ROOT / 'ablation_outputs' / 'retrieval'
DINOV3RAW_RETRIEVAL = RETRIEVAL_OUT / 'dinov3raw_retrieval_manifest.jsonl'
JOCCH_RETRIEVAL     = RETRIEVAL_OUT / 'jocch_retrieval_manifest.jsonl'

# Existing SSCD/CLIP geometry-filter outputs (from geometry_filter_ablation_colab.ipynb) --
# the real, sidecar-bearing sources for the new reuse-manifest mechanism.
GF_OUT = DRIVE_ROOT / 'ablation_outputs' / 'geometry_filter'
SSCD_S1_DIR = GF_OUT / 'sscd_shard1'
SSCD_S2_DIR = GF_OUT / 'sscd_shard2'
CLIP_S1_DIR = GF_OUT / 'clip_shard1'
CLIP_S2_DIR = GF_OUT / 'clip_shard2'

DRIVE_OUT_GF   = GF_OUT
DRIVE_OUT_VGGT = DRIVE_ROOT / 'ablation_outputs' / 'vggt'
BREAKPOINT = 50  # keypoint threshold (must match original pipeline)
# ────────────────────────────────────────────────────────────────────────────

LOCAL_WORK = Path('/content/dinov3raw_jocch_ablation')
(LOCAL_WORK / '_local').mkdir(parents=True, exist_ok=True)
DRIVE_OUT_GF.mkdir(parents=True, exist_ok=True)
DRIVE_OUT_VGGT.mkdir(parents=True, exist_ok=True)

SHARD1_CACHE = LOCAL_WORK / '_local' / 'aspan_all_manifest_shard1_reconstructed.jsonl'
SHARD2_CACHE = LOCAL_WORK / '_local' / 'aspan_all_manifest_shard2_reconstructed.jsonl'
REUSE_MANIFEST_LOCAL = LOCAL_WORK / 'aspan_reuse_manifest.jsonl'
# Stage-3 (VGGT/pose) analog of REUSE_MANIFEST_LOCAL -- see pose_reuse_core.py.
POSE_REUSE_MANIFEST_LOCAL = LOCAL_WORK / 'pose_reuse_manifest.jsonl'


def sync_dir_to_drive(local_dir, drive_dir, filenames, sidecar_dirname=None):
    """Copy named manifest files (if present) to Drive, and incrementally sync a sidecar dir.

    Manifests are small -- overwritten in full on every call. Sidecar .npz files can
    number in the hundreds, so only files not already present at the Drive destination
    are copied, keeping repeat calls across --resume re-runs cheap.
    """
    local_dir = Path(local_dir)
    drive_dir = Path(drive_dir)
    drive_dir.mkdir(parents=True, exist_ok=True)
    for name in filenames:
        src = local_dir / name
        if src.exists():
            shutil.copy2(src, drive_dir / name)
            print(f'  Saved: {name}')
        else:
            print(f'  MISSING (not yet produced): {name}')
    if sidecar_dirname:
        src_dir = local_dir / sidecar_dirname
        if src_dir.exists():
            dst_dir = drive_dir / sidecar_dirname
            dst_dir.mkdir(parents=True, exist_ok=True)
            all_npz = list(src_dir.glob('*.npz'))
            n_new = 0
            for npz in all_npz:
                dst = dst_dir / npz.name
                if not dst.exists():
                    shutil.copy2(npz, dst)
                    n_new += 1
            print(f'  Sidecars synced: {n_new} new ({len(all_npz)} total)')
        else:
            print(f'  MISSING (not yet produced): {sidecar_dirname}/')


print(f"ASpanFormer path     : {ASPANPATH}")
print(f"Weights              : {WEIGHTS_PATH}  (exists: {WEIGHTS_PATH.exists()})")
print(f"VGGT checkpoint      : {VGGT_CHECKPOINT}  (exists: {VGGT_CHECKPOINT.exists()})")
print(f"Source zip           : {SOURCE_ZIP}  (exists: {SOURCE_ZIP.exists()})")
print(f"Target zip           : {TARGET_ZIP}  (exists: {TARGET_ZIP.exists()})")
print(f"DINOv3-raw retrieval : {DINOV3RAW_RETRIEVAL}  (exists: {DINOV3RAW_RETRIEVAL.exists()})")
print(f"JOCCH retrieval      : {JOCCH_RETRIEVAL}  (exists: {JOCCH_RETRIEVAL.exists()})")
print(f"SSCD shard1 GF out   : {SSCD_S1_DIR}  (exists: {SSCD_S1_DIR.exists()})")
print(f"SSCD shard2 GF out   : {SSCD_S2_DIR}  (exists: {SSCD_S2_DIR.exists()})")
print(f"CLIP shard1 GF out   : {CLIP_S1_DIR}  (exists: {CLIP_S1_DIR.exists()})")
print(f"CLIP shard2 GF out   : {CLIP_S2_DIR}  (exists: {CLIP_S2_DIR.exists()})")

In [ ]:
# Step 3 — Copy scripts and cache manifests from Drive to Colab local filesystem
to_copy = [
    (SCRIPTS_DIR / 'geometry_filter.py',              LOCAL_WORK / 'geometry_filter.py'),
    (SCRIPTS_DIR / 'geometry_filter_cached.py',        LOCAL_WORK / 'geometry_filter_cached.py'),
    (SCRIPTS_DIR / 'build_aspan_reuse_manifest.py',    LOCAL_WORK / 'build_aspan_reuse_manifest.py'),
    (SCRIPTS_DIR / 'pose_reuse_core.py',               LOCAL_WORK / 'pose_reuse_core.py'),
    (SCRIPTS_DIR / 'build_pose_reuse_manifest.py',     LOCAL_WORK / 'build_pose_reuse_manifest.py'),
    (SCRIPTS_DIR / 'apply_pose_reuse.py',              LOCAL_WORK / 'apply_pose_reuse.py'),
    (SCRIPTS_DIR / 'vggt_signals.py',                  LOCAL_WORK / 'vggt_signals.py'),
    (SCRIPTS_DIR / 'pose_scoring.py',                  LOCAL_WORK / 'pose_scoring.py'),
    (SCRIPTS_DIR / '_local' / 'aspan_all_manifest_shard1_reconstructed.jsonl', SHARD1_CACHE),
    (SCRIPTS_DIR / '_local' / 'aspan_all_manifest_shard2_reconstructed.jsonl', SHARD2_CACHE),
]

for src, dst in to_copy:
    if not src.exists():
        raise FileNotFoundError(f"Missing on Drive: {src}\nUpload it to {src.parent} and re-run.")
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(str(src), str(dst))
    print(f"Copied: {src.name}  ({src.stat().st_size / 1e6:.2f} MB)")

sys.path.insert(0, str(LOCAL_WORK))
print("\nAll scripts and cache manifests ready.")

In [ ]:
# Step 3a — Copy ml-aspanformer from Drive
ASPAN_SRC = SCRIPTS_DIR / 'ml-aspanformer'

if not Path(ASPANPATH).exists():
    if not ASPAN_SRC.exists():
        raise FileNotFoundError(
            f"ml-aspanformer not found on Drive at {ASPAN_SRC}\n"
            "Upload the ml-aspanformer/ directory there and re-run."
        )
    print(f"Copying ml-aspanformer from Drive -> {ASPANPATH} ...")
    shutil.copytree(str(ASPAN_SRC), ASPANPATH)
    print("Done.")
else:
    print(f"ml-aspanformer already present at {ASPANPATH}")

In [ ]:
# Step 3b — Extract and flatten image corpus from Drive zips (idempotent)
import time

def _extract_and_flatten(zip_src, local_name):
    dst_dir = Path(f'./{local_name}')
    if dst_dir.exists() and any(dst_dir.iterdir()):
        n = sum(1 for p in dst_dir.iterdir() if p.is_file())
        print(f"  {local_name}/: already present ({n:,} files)")
        return n
    local_zip = Path(f'./{local_name}.zip')
    print(f"  Copying {Path(str(zip_src)).name} from Drive ...")
    shutil.copy(str(zip_src), str(local_zip))
    t0 = time.time()
    os.system(f'unzip -q {local_zip} -d {local_name}')
    print(f"  Unzipped in {time.time() - t0:.0f}s")
    for root, dirs, files in os.walk(str(dst_dir)):
        if root == str(dst_dir):
            continue
        for fname in files:
            src_path = os.path.join(root, fname)
            dst_path = os.path.join(str(dst_dir), fname)
            if os.path.exists(dst_path):
                base, ext = os.path.splitext(fname)
                dst_path = os.path.join(str(dst_dir), f"{base}_{root.replace('/', '_')}{ext}")
            shutil.move(src_path, dst_path)
    for root, dirs, files in os.walk(str(dst_dir), topdown=False):
        if root == str(dst_dir):
            continue
        try:
            os.rmdir(root)
        except OSError:
            pass
    local_zip.unlink(missing_ok=True)
    n = sum(1 for p in dst_dir.iterdir() if p.is_file())
    print(f"  {local_name}/: {n:,} files ready")
    return n

n_src = _extract_and_flatten(SOURCE_ZIP, 'source')
n_tgt = _extract_and_flatten(TARGET_ZIP, 'target')
print(f"\nImages ready: {n_src:,} sources, {n_tgt:,} targets")

In [ ]:
# Step 4 — Determine shard membership and define manifest splitter
# (identical logic to geometry_filter_ablation_colab.ipynb's cell-shard-split)

def collect_source_ids(jsonl_path):
    ids = set()
    with open(str(jsonl_path), encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                row = json.loads(line)
                sid = row.get('source_id')
                if sid:
                    ids.add(str(sid))
    return ids

shard1_sources = collect_source_ids(SHARD1_CACHE)
shard2_sources = collect_source_ids(SHARD2_CACHE)
print(f"Shard 1: {len(shard1_sources)} unique source IDs")
print(f"Shard 2: {len(shard2_sources)} unique source IDs")

overlap = shard1_sources & shard2_sources
if overlap:
    print(f"WARNING: {len(overlap)} source IDs appear in both shards — shard 1 wins for those.")


def _rewrite_path(p, local_root):
    path = Path(p)
    if path.is_absolute():
        return p
    return str(local_root / path.name)


def split_manifest_by_shard(manifest_path, shard_sources, out_path):
    rows = []
    with open(str(manifest_path), encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            if str(row.get('source_id', '')) not in shard_sources:
                continue
            if 'source_path' in row:
                row['source_path'] = _rewrite_path(row['source_path'], LOCAL_SOURCE_DIR.resolve())
            if 'target_path' in row:
                row['target_path'] = _rewrite_path(row['target_path'], LOCAL_TARGET_DIR.resolve())
            rows.append(json.dumps(row, ensure_ascii=False))
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text('\n'.join(rows) + ('\n' if rows else ''), encoding='utf-8')
    return len(rows)


def filter_new_pairs(src_manifest, dst_manifest, sidecar_local_dir, label):
    """Keep only new_pair=True rows and rewrite source/target/sidecar paths to this
    session's local filesystem.

    sidecar_path is rewritten to the local sidecar dir ONLY for fresh/legacy-cache rows,
    whose sidecar really does live there. cache_source == 'reuse_manifest' rows are
    EXCLUDED from that rewrite: their sidecar is a real file living in a *different*
    model's own directory (e.g. sscd_shard1/aspan_sidecars/), already a correct,
    resolvable absolute path -- rewriting it here would point at a file this session's
    local sidecar dir never had. (This was a real, confirmed bug on DINOv3-raw Shard 1 --
    32/98 new-pair candidates errored with FileNotFoundError until fixed. See
    project_dinov3raw_jocch_geometry_pipeline memory.) Most reuse_manifest rows never
    reach this function's fresh-VGGT path anyway -- apply_pose_reuse.py pre-seeds a
    borrowed judgment for them before vggt_signals.main() runs, so only the rare
    no-pose-hit fallback case actually depends on this fix.
    """
    rows = []
    n_total = 0
    with open(str(src_manifest), encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            n_total += 1
            if not row.get('new_pair'):
                continue
            if 'source_path' in row:
                row['source_path'] = str(LOCAL_SOURCE_DIR.resolve() / Path(row['source_path']).name)
            if 'target_path' in row:
                row['target_path'] = str(LOCAL_TARGET_DIR.resolve() / Path(row['target_path']).name)
            if 'sidecar_path' in row and row.get('cache_source') != 'reuse_manifest':
                row['sidecar_path'] = str(Path(sidecar_local_dir) / Path(row['sidecar_path']).name)
            rows.append(json.dumps(row, ensure_ascii=False))
    Path(str(dst_manifest)).parent.mkdir(parents=True, exist_ok=True)
    Path(str(dst_manifest)).write_text('\n'.join(rows) + ('\n' if rows else ''), encoding='utf-8')
    print(f"  {label}: {len(rows)}/{n_total} rows kept (new_pair=True)")
    if rows:
        r = json.loads(rows[0])
        print(f"    sample sidecar: {r.get('sidecar_path')}  (exists: {Path(str(r.get('sidecar_path', ''))).exists()})")
    return len(rows)


print("\nShard splitter and new_pair filter ready.")

In [ ]:
# Step 4a — Install vggt_omega via pip
# TODO: replace the URL below with the actual git URL you used previously
VGGT_GIT_URL = "git+https://github.com/TODO/vggt-omega.git"  # <- EDIT THIS

import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", VGGT_GIT_URL, "-q"],
    capture_output=True, text=True
)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f"pip install failed for {VGGT_GIT_URL}")

from vggt_omega.models import VGGTOmega
print("vggt_omega imported successfully.")

In [ ]:
# Step 5 — Reuse-manifest builders: rebuilt before each stage below, source lists growing.
# Two independent mechanisms: Stage 2 (ASpanFormer sidecar reuse, existing) and Stage 3
# (VGGT/pose judgment reuse, new) -- both Stage-1-ablation-only, see pose_reuse_core.py.
import importlib
import build_aspan_reuse_manifest, geometry_filter_cached, vggt_signals, pose_scoring
import pose_reuse_core, build_pose_reuse_manifest, apply_pose_reuse
importlib.reload(build_aspan_reuse_manifest)
importlib.reload(geometry_filter_cached)
importlib.reload(vggt_signals)
importlib.reload(pose_scoring)
importlib.reload(pose_reuse_core)
importlib.reload(build_pose_reuse_manifest)
importlib.reload(apply_pose_reuse)

# Verify the loaded geometry_filter_cached actually has the new_pair fix (2026-07-14).
# Checks the LIVE in-memory module via inspect.getsource(), not just the file on disk --
# this catches "forgot to importlib.reload() after re-uploading" as well as "forgot to
# re-upload to Drive at all", since a stale import would still show the old source here
# even if the file on disk is already correct.
import inspect
_gfc_src = inspect.getsource(geometry_filter_cached)
if '"new_pair": False,' in _gfc_src:
    raise RuntimeError(
        "geometry_filter_cached.py is STILL THE BUGGY VERSION -- it hardcodes "
        "\"new_pair\": False in at least one branch instead of using the computed "
        f"new_pair variable. Loaded from: {geometry_filter_cached.__file__}\n"
        "Fix: re-upload the corrected _local/geometry_filter_cached.py to Drive "
        "scripts/, re-run cell-copy-scripts, then re-run this cell."
    )
if 'new_pair = cache_key not in all_cached_pairs' not in _gfc_src:
    raise RuntimeError(
        "geometry_filter_cached.py's new_pair computation doesn't match the expected "
        f"fixed form (new_pair = cache_key not in all_cached_pairs). Loaded from: "
        f"{geometry_filter_cached.__file__}"
    )
print(f"geometry_filter_cached: new_pair fix verified present.")
print(f"  Loaded from: {geometry_filter_cached.__file__}")

# Same live-source guard for the filter_new_pairs() sidecar-path fix (2026-07-16) --
# defined in this notebook itself (cell-shard-split), not an imported module, so check
# the function's own source directly.
_fnp_src = inspect.getsource(filter_new_pairs)
if "row.get('cache_source') != 'reuse_manifest'" not in _fnp_src:
    raise RuntimeError(
        "filter_new_pairs() in this notebook is STILL THE BUGGY VERSION -- it "
        "unconditionally rewrites sidecar_path, which breaks cache_source=='reuse_manifest' "
        "rows whose sidecar lives in a different model's own directory. Re-run cell-shard-split "
        "with the fixed version before continuing."
    )
print(f"filter_new_pairs: sidecar-path fix verified present.")

# Starts with just the four SSCD/CLIP entries -- missing ones (shard2, not run yet) are
# loudly skipped by build_aspan_reuse_manifest.py itself, not silently ignored here.
completed_sources = [
    ('sscd_shard1', SSCD_S1_DIR / 'aspan_all_manifest.jsonl'),
    ('sscd_shard2', SSCD_S2_DIR / 'aspan_all_manifest.jsonl'),
    ('clip_shard1', CLIP_S1_DIR / 'aspan_all_manifest.jsonl'),
    ('clip_shard2', CLIP_S2_DIR / 'aspan_all_manifest.jsonl'),
]

# Stage-3 (VGGT/pose) analog -- sourced from each Stage-1 model's own completed
# pose_scored_manifest.jsonl instead of aspan_all_manifest.jsonl.
completed_pose_sources = [
    ('sscd_shard1', DRIVE_OUT_VGGT / 'sscd_shard1' / 'pose_scored_manifest.jsonl'),
    ('sscd_shard2', DRIVE_OUT_VGGT / 'sscd_shard2' / 'pose_scored_manifest.jsonl'),
    ('clip_shard1', DRIVE_OUT_VGGT / 'clip_shard1' / 'pose_scored_manifest.jsonl'),
    ('clip_shard2', DRIVE_OUT_VGGT / 'clip_shard2' / 'pose_scored_manifest.jsonl'),
]


def rebuild_reuse_manifest():
    """Re-consolidate every completed source (SSCD/CLIP + whatever this run has already
    produced) into REUSE_MANIFEST_LOCAL, then sync it to Drive immediately. Cheap: JSONL
    reads + one directory listing per source, no GPU/compute work."""
    sources_arg = [f'{tag}={path}' for tag, path in completed_sources]
    build_aspan_reuse_manifest.main(['--sources', *sources_arg, '--output', str(REUSE_MANIFEST_LOCAL)])
    sync_dir_to_drive(LOCAL_WORK, DRIVE_OUT_GF, ['aspan_reuse_manifest.jsonl'])


def rebuild_pose_reuse_manifest():
    """Stage-3 analog of rebuild_reuse_manifest() -- re-consolidate every completed
    pose_scored_manifest.jsonl source into POSE_REUSE_MANIFEST_LOCAL. Also cheap:
    JSONL reads only, no GPU/compute work."""
    sources_arg = [f'{tag}={path}' for tag, path in completed_pose_sources]
    build_pose_reuse_manifest.main(['--sources', *sources_arg, '--output', str(POSE_REUSE_MANIFEST_LOCAL)])
    sync_dir_to_drive(LOCAL_WORK, DRIVE_OUT_GF, ['pose_reuse_manifest.jsonl'])


rebuild_reuse_manifest()
rebuild_pose_reuse_manifest()

In [ ]:
# Step 6 — Smoke test: confirm the reuse-manifest mechanism actually works before
# committing to the full run. Uses --max-pairs on whichever retrieval manifest exists.
smoke_input = DINOV3RAW_RETRIEVAL if DINOV3RAW_RETRIEVAL.exists() else JOCCH_RETRIEVAL
if not smoke_input.exists():
    print("SKIPPING smoke test: neither DINOv3-raw nor JOCCH retrieval manifest found yet.")
else:
    SMOKE_INPUT = LOCAL_WORK / 'smoke_candidates.jsonl'
    n_smoke = split_manifest_by_shard(smoke_input, shard1_sources, SMOKE_INPUT)
    print(f"Smoke test input: {n_smoke} candidate rows (capped further by --max-pairs)")

    SMOKE_OUT = LOCAL_WORK / 'smoke_output'
    geometry_filter_cached.main([
        '--input-manifest',   str(SMOKE_INPUT),
        '--output-dir',       str(SMOKE_OUT),
        '--breakpoint-value', str(BREAKPOINT),
        '--aspanpath',        ASPANPATH,
        '--weights_path',     str(WEIGHTS_PATH),
        '--config_path',      CONFIG_PATH,
        '--cache-manifests',  str(SHARD1_CACHE),
        '--reuse-manifests',  str(REUSE_MANIFEST_LOCAL),
        '--device',           'auto',
        '--max-pairs',        '25',
        '--progress-every',   '5',
    ])

    smoke_all = SMOKE_OUT / 'aspan_all_manifest.jsonl'
    if smoke_all.exists():
        rows = [json.loads(l) for l in smoke_all.read_text(encoding='utf-8').splitlines() if l.strip()]
        n_reuse = sum(1 for r in rows if r.get('cache_source') == 'reuse_manifest')
        print(f"\nSmoke test: {len(rows)} candidates checked, {n_reuse} hit the reuse manifest "
              f"(0 is possible by chance on a small random sample -- not itself a failure, "
              f"but worth noting before the full run).")
    print("\nSmoke test passed.")

## DINOv3-raw — Shard 1

Geometry filter (with cache + reuse) -> VGGT signals -> pose scoring, each stage synced to Drive immediately after it completes.

In [ ]:
# DINOv3-raw — Shard 1 — Geometry Filter
DINOV3RAW_S1_INPUT  = LOCAL_WORK / 'dinov3raw_s1_candidates.jsonl'
DINOV3RAW_S1_OUTPUT = LOCAL_WORK / 'dinov3raw_s1_output'

if not DINOV3RAW_RETRIEVAL.exists():
    print(f"SKIPPING: {DINOV3RAW_RETRIEVAL} not found yet -- run retrieval_ablation_colab.ipynb first.")
else:
    n = split_manifest_by_shard(DINOV3RAW_RETRIEVAL, shard1_sources, DINOV3RAW_S1_INPUT)
    print(f"DINOv3-raw shard 1 input: {n} candidate rows")

    geometry_filter_cached.main([
        '--input-manifest',   str(DINOV3RAW_S1_INPUT),
        '--output-dir',       str(DINOV3RAW_S1_OUTPUT),
        '--breakpoint-value', str(BREAKPOINT),
        '--aspanpath',        ASPANPATH,
        '--weights_path',     str(WEIGHTS_PATH),
        '--config_path',      CONFIG_PATH,
        '--cache-manifests',  str(SHARD1_CACHE),
        '--reuse-manifests',  str(REUSE_MANIFEST_LOCAL),
        '--device',           'auto',
        '--progress-every',   '100',
    ])

    sync_dir_to_drive(DINOV3RAW_S1_OUTPUT, DRIVE_OUT_GF / 'dinov3raw_shard1',
                       ['aspan_all_manifest.jsonl', 'vggt_candidates_manifest.jsonl'],
                       sidecar_dirname='aspan_sidecars')

    print(f"\nDINOv3-raw shard 1 geometry filter complete. Output: {DINOV3RAW_S1_OUTPUT}")

In [ ]:
# DINOv3-raw — Shard 1 — VGGT signals + Pose scoring (new_pair=True only)
DINOV3RAW_S1_NEW_MANIFEST = LOCAL_WORK / 'dinov3raw_s1_new_pairs.jsonl'
DINOV3RAW_S1_VGGT_OUT = LOCAL_WORK / 'dinov3raw_s1_vggt_output'
DINOV3RAW_S1_POSE_OUT = LOCAL_WORK / 'dinov3raw_s1_pose_output'

if not DINOV3RAW_RETRIEVAL.exists():
    print("SKIPPING (no retrieval manifest for this model yet).")
    n_dinov3raw_s1_new = 0
else:
    n_dinov3raw_s1_new = filter_new_pairs(
        DINOV3RAW_S1_OUTPUT / 'vggt_candidates_manifest.jsonl',
        DINOV3RAW_S1_NEW_MANIFEST,
        DINOV3RAW_S1_OUTPUT / 'aspan_sidecars',
        'DINOv3-raw shard 1',
    )

    if n_dinov3raw_s1_new == 0:
        print("No new pairs -- skipping VGGT.")
    else:
        # Pre-seed judged/true-match manifests with cross-model borrowed judgments
        # (Stage 1 reuse only) BEFORE vggt_signals.main(--resume) runs -- its own
        # --resume mechanism then automatically skips the pre-seeded candidate_ids and
        # only computes VGGT fresh for the genuine remainder.
        apply_pose_reuse.main([
            '--new-pairs-manifest',    str(DINOV3RAW_S1_NEW_MANIFEST),
            '--pose-reuse-manifest',   str(POSE_REUSE_MANIFEST_LOCAL),
            '--vggt-judged-manifest',  str(DINOV3RAW_S1_VGGT_OUT / 'vggt_judged_manifest.jsonl'),
            '--true-matches-manifest', str(DINOV3RAW_S1_VGGT_OUT / 'true_matches_manifest.jsonl'),
        ])
        vggt_signals.main([
            '--input-manifest', str(DINOV3RAW_S1_NEW_MANIFEST),
            '--output-dir',     str(DINOV3RAW_S1_VGGT_OUT),
            '--checkpoint',     str(VGGT_CHECKPOINT),
            '--device',         'auto',
            '--progress-every', '5',
            '--resume',
        ])
    sync_dir_to_drive(DINOV3RAW_S1_VGGT_OUT, DRIVE_OUT_VGGT / 'dinov3raw_shard1',
                       ['vggt_judged_manifest.jsonl', 'true_matches_manifest.jsonl', 'vggt_judge_summary.json'])

In [ ]:
# DINOv3-raw — Shard 1 — Pose scoring
if not DINOV3RAW_RETRIEVAL.exists() or n_dinov3raw_s1_new == 0:
    print("SKIPPING pose scoring (no new pairs / retrieval manifest missing).")
else:
    pose_scoring.main([
        '--input-manifest', str(DINOV3RAW_S1_VGGT_OUT / 'vggt_judged_manifest.jsonl'),
        '--output-dir',     str(DINOV3RAW_S1_POSE_OUT),
    ])
sync_dir_to_drive(DINOV3RAW_S1_POSE_OUT, DRIVE_OUT_VGGT / 'dinov3raw_shard1',
                   ['pose_scored_manifest.jsonl', 'pose_scoring_summary.json'])

In [ ]:
# Rebuild the reuse manifests, folding in DINOv3-raw shard 1's own output (if it ran)
# before the next stage, so later stages benefit from it too.
if DINOV3RAW_S1_OUTPUT.exists() and (DINOV3RAW_S1_OUTPUT / 'aspan_all_manifest.jsonl').exists():
    completed_sources.append(('dinov3raw_shard1', DINOV3RAW_S1_OUTPUT / 'aspan_all_manifest.jsonl'))
    print(f"Added 'dinov3raw_shard1' to completed_sources.")
else:
    print(f"'dinov3raw_shard1' did not produce output -- not added to completed_sources.")
rebuild_reuse_manifest()

if DINOV3RAW_S1_POSE_OUT.exists() and (DINOV3RAW_S1_POSE_OUT / 'pose_scored_manifest.jsonl').exists():
    completed_pose_sources.append(('dinov3raw_shard1', DINOV3RAW_S1_POSE_OUT / 'pose_scored_manifest.jsonl'))
    print(f"Added 'dinov3raw_shard1' to completed_pose_sources.")
else:
    print(f"'dinov3raw_shard1' produced no pose_scored_manifest -- not added to completed_pose_sources.")
rebuild_pose_reuse_manifest()

## DINOv3-raw — Shard 2

Geometry filter (with cache + reuse) -> VGGT signals -> pose scoring, each stage synced to Drive immediately after it completes.

In [ ]:
# DINOv3-raw — Shard 2 — Geometry Filter
DINOV3RAW_S2_INPUT  = LOCAL_WORK / 'dinov3raw_s2_candidates.jsonl'
DINOV3RAW_S2_OUTPUT = LOCAL_WORK / 'dinov3raw_s2_output'

if not DINOV3RAW_RETRIEVAL.exists():
    print(f"SKIPPING: {DINOV3RAW_RETRIEVAL} not found yet -- run retrieval_ablation_colab.ipynb first.")
else:
    n = split_manifest_by_shard(DINOV3RAW_RETRIEVAL, shard2_sources, DINOV3RAW_S2_INPUT)
    print(f"DINOv3-raw shard 2 input: {n} candidate rows")

    geometry_filter_cached.main([
        '--input-manifest',   str(DINOV3RAW_S2_INPUT),
        '--output-dir',       str(DINOV3RAW_S2_OUTPUT),
        '--breakpoint-value', str(BREAKPOINT),
        '--aspanpath',        ASPANPATH,
        '--weights_path',     str(WEIGHTS_PATH),
        '--config_path',      CONFIG_PATH,
        '--cache-manifests',  str(SHARD2_CACHE),
        '--reuse-manifests',  str(REUSE_MANIFEST_LOCAL),
        '--device',           'auto',
        '--progress-every',   '100',
    ])

    sync_dir_to_drive(DINOV3RAW_S2_OUTPUT, DRIVE_OUT_GF / 'dinov3raw_shard2',
                       ['aspan_all_manifest.jsonl', 'vggt_candidates_manifest.jsonl'],
                       sidecar_dirname='aspan_sidecars')

    print(f"\nDINOv3-raw shard 2 geometry filter complete. Output: {DINOV3RAW_S2_OUTPUT}")

In [ ]:
# DINOv3-raw — Shard 2 — VGGT signals + Pose scoring (new_pair=True only)
DINOV3RAW_S2_NEW_MANIFEST = LOCAL_WORK / 'dinov3raw_s2_new_pairs.jsonl'
DINOV3RAW_S2_VGGT_OUT = LOCAL_WORK / 'dinov3raw_s2_vggt_output'
DINOV3RAW_S2_POSE_OUT = LOCAL_WORK / 'dinov3raw_s2_pose_output'

if not DINOV3RAW_RETRIEVAL.exists():
    print("SKIPPING (no retrieval manifest for this model yet).")
    n_dinov3raw_s2_new = 0
else:
    n_dinov3raw_s2_new = filter_new_pairs(
        DINOV3RAW_S2_OUTPUT / 'vggt_candidates_manifest.jsonl',
        DINOV3RAW_S2_NEW_MANIFEST,
        DINOV3RAW_S2_OUTPUT / 'aspan_sidecars',
        'DINOv3-raw shard 2',
    )

    if n_dinov3raw_s2_new == 0:
        print("No new pairs -- skipping VGGT.")
    else:
        apply_pose_reuse.main([
            '--new-pairs-manifest',    str(DINOV3RAW_S2_NEW_MANIFEST),
            '--pose-reuse-manifest',   str(POSE_REUSE_MANIFEST_LOCAL),
            '--vggt-judged-manifest',  str(DINOV3RAW_S2_VGGT_OUT / 'vggt_judged_manifest.jsonl'),
            '--true-matches-manifest', str(DINOV3RAW_S2_VGGT_OUT / 'true_matches_manifest.jsonl'),
        ])
        vggt_signals.main([
            '--input-manifest', str(DINOV3RAW_S2_NEW_MANIFEST),
            '--output-dir',     str(DINOV3RAW_S2_VGGT_OUT),
            '--checkpoint',     str(VGGT_CHECKPOINT),
            '--device',         'auto',
            '--progress-every', '5',
            '--resume',
        ])
    sync_dir_to_drive(DINOV3RAW_S2_VGGT_OUT, DRIVE_OUT_VGGT / 'dinov3raw_shard2',
                       ['vggt_judged_manifest.jsonl', 'true_matches_manifest.jsonl', 'vggt_judge_summary.json'])

In [ ]:
# DINOv3-raw — Shard 2 — Pose scoring
if not DINOV3RAW_RETRIEVAL.exists() or n_dinov3raw_s2_new == 0:
    print("SKIPPING pose scoring (no new pairs / retrieval manifest missing).")
else:
    pose_scoring.main([
        '--input-manifest', str(DINOV3RAW_S2_VGGT_OUT / 'vggt_judged_manifest.jsonl'),
        '--output-dir',     str(DINOV3RAW_S2_POSE_OUT),
    ])
sync_dir_to_drive(DINOV3RAW_S2_POSE_OUT, DRIVE_OUT_VGGT / 'dinov3raw_shard2',
                   ['pose_scored_manifest.jsonl', 'pose_scoring_summary.json'])

In [ ]:
# Rebuild the reuse manifests, folding in DINOv3-raw shard 2's own output (if it ran)
# before the next stage, so later stages benefit from it too.
if DINOV3RAW_S2_OUTPUT.exists() and (DINOV3RAW_S2_OUTPUT / 'aspan_all_manifest.jsonl').exists():
    completed_sources.append(('dinov3raw_shard2', DINOV3RAW_S2_OUTPUT / 'aspan_all_manifest.jsonl'))
    print(f"Added 'dinov3raw_shard2' to completed_sources.")
else:
    print(f"'dinov3raw_shard2' did not produce output -- not added to completed_sources.")
rebuild_reuse_manifest()

if DINOV3RAW_S2_POSE_OUT.exists() and (DINOV3RAW_S2_POSE_OUT / 'pose_scored_manifest.jsonl').exists():
    completed_pose_sources.append(('dinov3raw_shard2', DINOV3RAW_S2_POSE_OUT / 'pose_scored_manifest.jsonl'))
    print(f"Added 'dinov3raw_shard2' to completed_pose_sources.")
else:
    print(f"'dinov3raw_shard2' produced no pose_scored_manifest -- not added to completed_pose_sources.")
rebuild_pose_reuse_manifest()

## JOCCH — Shard 1

Geometry filter (with cache + reuse) -> VGGT signals -> pose scoring, each stage synced to Drive immediately after it completes.

In [ ]:
# JOCCH — Shard 1 — Geometry Filter
JOCCH_S1_INPUT  = LOCAL_WORK / 'jocch_s1_candidates.jsonl'
JOCCH_S1_OUTPUT = LOCAL_WORK / 'jocch_s1_output'

if not JOCCH_RETRIEVAL.exists():
    print(f"SKIPPING: {JOCCH_RETRIEVAL} not found yet -- run retrieval_ablation_colab.ipynb first.")
else:
    n = split_manifest_by_shard(JOCCH_RETRIEVAL, shard1_sources, JOCCH_S1_INPUT)
    print(f"JOCCH shard 1 input: {n} candidate rows")

    geometry_filter_cached.main([
        '--input-manifest',   str(JOCCH_S1_INPUT),
        '--output-dir',       str(JOCCH_S1_OUTPUT),
        '--breakpoint-value', str(BREAKPOINT),
        '--aspanpath',        ASPANPATH,
        '--weights_path',     str(WEIGHTS_PATH),
        '--config_path',      CONFIG_PATH,
        '--cache-manifests',  str(SHARD1_CACHE),
        '--reuse-manifests',  str(REUSE_MANIFEST_LOCAL),
        '--device',           'auto',
        '--progress-every',   '100',
    ])

    sync_dir_to_drive(JOCCH_S1_OUTPUT, DRIVE_OUT_GF / 'jocch_shard1',
                       ['aspan_all_manifest.jsonl', 'vggt_candidates_manifest.jsonl'],
                       sidecar_dirname='aspan_sidecars')

    print(f"\nJOCCH shard 1 geometry filter complete. Output: {JOCCH_S1_OUTPUT}")

In [ ]:
# JOCCH — Shard 1 — VGGT signals + Pose scoring (new_pair=True only)
JOCCH_S1_NEW_MANIFEST = LOCAL_WORK / 'jocch_s1_new_pairs.jsonl'
JOCCH_S1_VGGT_OUT = LOCAL_WORK / 'jocch_s1_vggt_output'
JOCCH_S1_POSE_OUT = LOCAL_WORK / 'jocch_s1_pose_output'

if not JOCCH_RETRIEVAL.exists():
    print("SKIPPING (no retrieval manifest for this model yet).")
    n_jocch_s1_new = 0
else:
    n_jocch_s1_new = filter_new_pairs(
        JOCCH_S1_OUTPUT / 'vggt_candidates_manifest.jsonl',
        JOCCH_S1_NEW_MANIFEST,
        JOCCH_S1_OUTPUT / 'aspan_sidecars',
        'JOCCH shard 1',
    )

    if n_jocch_s1_new == 0:
        print("No new pairs -- skipping VGGT.")
    else:
        apply_pose_reuse.main([
            '--new-pairs-manifest',    str(JOCCH_S1_NEW_MANIFEST),
            '--pose-reuse-manifest',   str(POSE_REUSE_MANIFEST_LOCAL),
            '--vggt-judged-manifest',  str(JOCCH_S1_VGGT_OUT / 'vggt_judged_manifest.jsonl'),
            '--true-matches-manifest', str(JOCCH_S1_VGGT_OUT / 'true_matches_manifest.jsonl'),
        ])
        vggt_signals.main([
            '--input-manifest', str(JOCCH_S1_NEW_MANIFEST),
            '--output-dir',     str(JOCCH_S1_VGGT_OUT),
            '--checkpoint',     str(VGGT_CHECKPOINT),
            '--device',         'auto',
            '--progress-every', '5',
            '--resume',
        ])
    sync_dir_to_drive(JOCCH_S1_VGGT_OUT, DRIVE_OUT_VGGT / 'jocch_shard1',
                       ['vggt_judged_manifest.jsonl', 'true_matches_manifest.jsonl', 'vggt_judge_summary.json'])

In [ ]:
# JOCCH — Shard 1 — Pose scoring
if not JOCCH_RETRIEVAL.exists() or n_jocch_s1_new == 0:
    print("SKIPPING pose scoring (no new pairs / retrieval manifest missing).")
else:
    pose_scoring.main([
        '--input-manifest', str(JOCCH_S1_VGGT_OUT / 'vggt_judged_manifest.jsonl'),
        '--output-dir',     str(JOCCH_S1_POSE_OUT),
    ])
sync_dir_to_drive(JOCCH_S1_POSE_OUT, DRIVE_OUT_VGGT / 'jocch_shard1',
                   ['pose_scored_manifest.jsonl', 'pose_scoring_summary.json'])

In [ ]:
# Rebuild the reuse manifests, folding in JOCCH shard 1's own output (if it ran) before
# the next stage, so later stages benefit from it too.
if JOCCH_S1_OUTPUT.exists() and (JOCCH_S1_OUTPUT / 'aspan_all_manifest.jsonl').exists():
    completed_sources.append(('jocch_shard1', JOCCH_S1_OUTPUT / 'aspan_all_manifest.jsonl'))
    print(f"Added 'jocch_shard1' to completed_sources.")
else:
    print(f"'jocch_shard1' did not produce output -- not added to completed_sources.")
rebuild_reuse_manifest()

if JOCCH_S1_POSE_OUT.exists() and (JOCCH_S1_POSE_OUT / 'pose_scored_manifest.jsonl').exists():
    completed_pose_sources.append(('jocch_shard1', JOCCH_S1_POSE_OUT / 'pose_scored_manifest.jsonl'))
    print(f"Added 'jocch_shard1' to completed_pose_sources.")
else:
    print(f"'jocch_shard1' produced no pose_scored_manifest -- not added to completed_pose_sources.")
rebuild_pose_reuse_manifest()

## JOCCH — Shard 2

Geometry filter (with cache + reuse) -> VGGT signals -> pose scoring, each stage synced to Drive immediately after it completes.

In [ ]:
# JOCCH — Shard 2 — Geometry Filter
JOCCH_S2_INPUT  = LOCAL_WORK / 'jocch_s2_candidates.jsonl'
JOCCH_S2_OUTPUT = LOCAL_WORK / 'jocch_s2_output'

if not JOCCH_RETRIEVAL.exists():
    print(f"SKIPPING: {JOCCH_RETRIEVAL} not found yet -- run retrieval_ablation_colab.ipynb first.")
else:
    n = split_manifest_by_shard(JOCCH_RETRIEVAL, shard2_sources, JOCCH_S2_INPUT)
    print(f"JOCCH shard 2 input: {n} candidate rows")

    geometry_filter_cached.main([
        '--input-manifest',   str(JOCCH_S2_INPUT),
        '--output-dir',       str(JOCCH_S2_OUTPUT),
        '--breakpoint-value', str(BREAKPOINT),
        '--aspanpath',        ASPANPATH,
        '--weights_path',     str(WEIGHTS_PATH),
        '--config_path',      CONFIG_PATH,
        '--cache-manifests',  str(SHARD2_CACHE),
        '--reuse-manifests',  str(REUSE_MANIFEST_LOCAL),
        '--device',           'auto',
        '--progress-every',   '100',
    ])

    sync_dir_to_drive(JOCCH_S2_OUTPUT, DRIVE_OUT_GF / 'jocch_shard2',
                       ['aspan_all_manifest.jsonl', 'vggt_candidates_manifest.jsonl'],
                       sidecar_dirname='aspan_sidecars')

    print(f"\nJOCCH shard 2 geometry filter complete. Output: {JOCCH_S2_OUTPUT}")

In [ ]:
# JOCCH — Shard 2 — VGGT signals + Pose scoring (new_pair=True only)
JOCCH_S2_NEW_MANIFEST = LOCAL_WORK / 'jocch_s2_new_pairs.jsonl'
JOCCH_S2_VGGT_OUT = LOCAL_WORK / 'jocch_s2_vggt_output'
JOCCH_S2_POSE_OUT = LOCAL_WORK / 'jocch_s2_pose_output'

if not JOCCH_RETRIEVAL.exists():
    print("SKIPPING (no retrieval manifest for this model yet).")
    n_jocch_s2_new = 0
else:
    n_jocch_s2_new = filter_new_pairs(
        JOCCH_S2_OUTPUT / 'vggt_candidates_manifest.jsonl',
        JOCCH_S2_NEW_MANIFEST,
        JOCCH_S2_OUTPUT / 'aspan_sidecars',
        'JOCCH shard 2',
    )

    if n_jocch_s2_new == 0:
        print("No new pairs -- skipping VGGT.")
    else:
        apply_pose_reuse.main([
            '--new-pairs-manifest',    str(JOCCH_S2_NEW_MANIFEST),
            '--pose-reuse-manifest',   str(POSE_REUSE_MANIFEST_LOCAL),
            '--vggt-judged-manifest',  str(JOCCH_S2_VGGT_OUT / 'vggt_judged_manifest.jsonl'),
            '--true-matches-manifest', str(JOCCH_S2_VGGT_OUT / 'true_matches_manifest.jsonl'),
        ])
        vggt_signals.main([
            '--input-manifest', str(JOCCH_S2_NEW_MANIFEST),
            '--output-dir',     str(JOCCH_S2_VGGT_OUT),
            '--checkpoint',     str(VGGT_CHECKPOINT),
            '--device',         'auto',
            '--progress-every', '5',
            '--resume',
        ])
    sync_dir_to_drive(JOCCH_S2_VGGT_OUT, DRIVE_OUT_VGGT / 'jocch_shard2',
                       ['vggt_judged_manifest.jsonl', 'true_matches_manifest.jsonl', 'vggt_judge_summary.json'])

In [ ]:
# JOCCH — Shard 2 — Pose scoring
if not JOCCH_RETRIEVAL.exists() or n_jocch_s2_new == 0:
    print("SKIPPING pose scoring (no new pairs / retrieval manifest missing).")
else:
    pose_scoring.main([
        '--input-manifest', str(JOCCH_S2_VGGT_OUT / 'vggt_judged_manifest.jsonl'),
        '--output-dir',     str(JOCCH_S2_POSE_OUT),
    ])
sync_dir_to_drive(JOCCH_S2_POSE_OUT, DRIVE_OUT_VGGT / 'jocch_shard2',
                   ['pose_scored_manifest.jsonl', 'pose_scoring_summary.json'])

In [ ]:
# Rebuild the reuse manifests, folding in JOCCH shard 2's own output (if it ran) before
# the next stage, so later stages benefit from it too.
if JOCCH_S2_OUTPUT.exists() and (JOCCH_S2_OUTPUT / 'aspan_all_manifest.jsonl').exists():
    completed_sources.append(('jocch_shard2', JOCCH_S2_OUTPUT / 'aspan_all_manifest.jsonl'))
    print(f"Added 'jocch_shard2' to completed_sources.")
else:
    print(f"'jocch_shard2' did not produce output -- not added to completed_sources.")
rebuild_reuse_manifest()

if JOCCH_S2_POSE_OUT.exists() and (JOCCH_S2_POSE_OUT / 'pose_scored_manifest.jsonl').exists():
    completed_pose_sources.append(('jocch_shard2', JOCCH_S2_POSE_OUT / 'pose_scored_manifest.jsonl'))
    print(f"Added 'jocch_shard2' to completed_pose_sources.")
else:
    print(f"'jocch_shard2' produced no pose_scored_manifest -- not added to completed_pose_sources.")
rebuild_pose_reuse_manifest()

## Verify outputs

Prints row counts, cache/reuse hit rates, and a sample row for each model x shard combination, plus a Drive-output status table.

In [ ]:
# Verification / summary
def verify_gf(model_tag, shard_num, out_dir):
    out_dir = Path(str(out_dir))
    all_path = out_dir / 'aspan_all_manifest.jsonl'
    pass_path = out_dir / 'vggt_candidates_manifest.jsonl'
    if not all_path.exists():
        print(f"{model_tag} shard{shard_num}: all manifest not found (not run yet)\n")
        return
    all_rows = [json.loads(l) for l in all_path.read_text(encoding='utf-8').splitlines() if l.strip()]
    pass_rows = [json.loads(l) for l in pass_path.read_text(encoding='utf-8').splitlines() if l.strip()] \
                if pass_path.exists() else []
    n_total = len(all_rows)
    n_pass = len(pass_rows)
    n_cached = sum(1 for r in all_rows if r.get('from_cache'))
    n_reuse_pass = sum(1 for r in all_rows if r.get('cache_source') == 'reuse_manifest' and r.get('aspan_pass'))
    n_reuse_fail = sum(1 for r in all_rows if r.get('cache_source') == 'reuse_manifest' and not r.get('aspan_pass'))
    n_legacy_cache = n_cached - n_reuse_pass - n_reuse_fail
    n_new = sum(1 for r in all_rows if r.get('new_pair'))
    print(f"── {model_tag} shard{shard_num} {'─' * 35}")
    print(f"  All pairs         : {n_total:,}")
    print(f"  Passed filter     : {n_pass:,}  ({100*n_pass/max(n_total,1):.1f}%)")
    print(f"  Cache hits (total): {n_cached:,}  ({100*n_cached/max(n_total,1):.1f}% skipped ASpanFormer)")
    print(f"    reuse-manifest passes : {n_reuse_pass:,}  (real sidecar reused, no ASpanFormer run)")
    print(f"    reuse-manifest fails  : {n_reuse_fail:,}")
    print(f"    legacy DINO-cache     : {n_legacy_cache:,}")
    print(f"  New pairs         : {n_new:,}  ({100*n_new/max(n_total,1):.1f}% not seen anywhere before)")
    sidecar_dir = out_dir / 'aspan_sidecars'
    n_npz = len(list(sidecar_dir.glob('*.npz'))) if sidecar_dir.exists() else 0
    print(f"  Fresh sidecar NPZs: {n_npz:,}  (only for newly-run passes, reused ones aren't copied)")
    return all_rows


def verify_vggt(model_tag, shard_num, aspan_all_rows, vggt_out_dir):
    """Stage-3 (VGGT/pose) reuse reporting -- how many new-pair candidates were borrowed
    from a peer Stage-1 model's own judgment (no VGGT compute) vs. freshly judged, and
    how many reuse-sourced candidates had no pose-reuse hit and needed fresh VGGT anyway.
    """
    vggt_out_dir = Path(str(vggt_out_dir))
    pose_path = vggt_out_dir / 'pose_scored_manifest.jsonl'
    if not pose_path.exists():
        return
    pose_rows = [json.loads(l) for l in pose_path.read_text(encoding='utf-8').splitlines() if l.strip()]

    n_pose_total = len(pose_rows)
    borrowed_keys = set((r.get('source_id'), r.get('target_id')) for r in pose_rows if r.get('pose_reused_from'))
    n_borrowed = len(borrowed_keys)
    n_fresh = n_pose_total - n_borrowed

    reuse_sourced_new_pair_keys = set(
        (r.get('source_id'), r.get('target_id')) for r in aspan_all_rows
        if r.get('cache_source') == 'reuse_manifest' and r.get('new_pair')
    )
    pose_keys = set((r.get('source_id'), r.get('target_id')) for r in pose_rows)
    n_reuse_no_pose_hit = len((reuse_sourced_new_pair_keys & pose_keys) - borrowed_keys)

    print(f"  Stage 3 (VGGT/pose): {n_pose_total:,} judged, "
          f"{n_borrowed:,} borrowed (no VGGT compute), {n_fresh:,} freshly computed")
    print(f"  reuse-sourced new pairs with no pose-reuse hit (needed fresh VGGT anyway): {n_reuse_no_pose_hit:,}")
    print()


# Hardcoded here (not a reference to a notebook-level list) -- these are exactly the
# four model x shard combinations this notebook runs, in the same order as above.
verify_blocks = [
    ('dinov3raw', 'DINOv3-raw', 1),
    ('dinov3raw', 'DINOv3-raw', 2),
    ('jocch',     'JOCCH',      1),
    ('jocch',     'JOCCH',      2),
]

for tag, label, shard_num in verify_blocks:
    key = f"{tag}_s{shard_num}"
    out_var_name = f"{key.upper()}_OUTPUT"
    if out_var_name in globals():
        aspan_all_rows = verify_gf(label, shard_num, globals()[out_var_name])
        vggt_out_var_name = f"{tag.upper()}_S{shard_num}_POSE_OUT"
        if aspan_all_rows is not None and vggt_out_var_name in globals():
            verify_vggt(label, shard_num, aspan_all_rows, globals()[vggt_out_var_name])
        else:
            print()

print("── Drive output status ──────────────────────────────────────────")
for tag, label, shard_num in verify_blocks:
    key_dir = f"{tag}_shard{shard_num}"
    gf_dir = DRIVE_OUT_GF / key_dir
    vggt_dir = DRIVE_OUT_VGGT / key_dir
    print(f"{key_dir}:")
    for name in ['aspan_all_manifest.jsonl', 'vggt_candidates_manifest.jsonl']:
        p = gf_dir / name
        print(f"  {name}: {'OK' if p.exists() else 'missing'}")
    sidecar_dir = gf_dir / 'aspan_sidecars'
    n_npz = len(list(sidecar_dir.glob('*.npz'))) if sidecar_dir.exists() else 0
    print(f"  aspan_sidecars/: {n_npz} npz files")
    for name in ['vggt_judged_manifest.jsonl', 'pose_scored_manifest.jsonl', 'pose_scoring_summary.json']:
        p = vggt_dir / name
        print(f"  vggt/{name}: {'OK' if p.exists() else 'missing'}")
    print()